# Chapter 11 &mdash; Growing "Inside-Out" by Rearrangement

**Concept 21 of the Chapter 11 decomposition:** *Growing "Inside-Out" by Rearrangement: Building the Grammar for $\overline{L_{ww}}$*

Re-view $p\,0\,q\ p'\,1\,q'$ as $(p\,0\,p')(q\,1\,q')$ &mdash; legal because only the lengths matter.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-Rearrangement-Trick/Concept-Rearrangement-Trick.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Here is how the grammar of Concept 20 is actually found.

A non-$ww$ even-length string has a mismatch at some position: writing the halves as
$p\,c\,q$ and $p'\,c'\,q'$ with $|p|=|p'|$, $|q|=|q'|$ and $c\ne c'$, the string is

$$p\,c\,q\;p'\,c'\,q'.$$

That is hard to generate directly &mdash; the constraint links symbols far apart. **The
rearrangement:** since only the *lengths* are constrained, regroup as

$$(p\,c\,p')\ (q\,c'\,q')$$

with $|p|=|p'|$ and $|q|=|q'|$. Each group is now an **odd-length string with a known
centre symbol** &mdash; an onion! And that is exactly what `A` and `B` generate.

The trick is worth remembering: **when a constraint is awkward, look for an equivalent
regrouping in which it becomes local.**

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### The two nonterminals the rearrangement produces

In [ ]:
A_ = mkg({'A': ["a", "aAa", "aAb", "bAa", "bAb"]}, 'A')   # odd, centre 'a'
B_ = mkg({'B': ["b", "aBa", "aBb", "bBa", "bBb"]}, 'B')   # odd, centre 'b'
NotWW = mkg({'S': ["A", "B", "AB", "BA"],
             'A': ["a", "aAa", "aAb", "bAa", "bAb"],
             'B': ["b", "aBa", "aBb", "bBa", "bBb"]})

### The rearrangement, performed explicitly

In [ ]:
def rearrange(s):
    # s is even-length and not ww: find a mismatching position and regroup
    h = len(s) // 2
    ps = [i for i in range(h) if s[i] != s[h+i]]
    if not ps: return None
    i = ps[0]
    p,  c,  q  = s[:i],     s[i],     s[i+1:h]
    p2, c2, q2 = s[h:h+i],  s[h+i],   s[h+i+1:]
    return (p + c + p2, q + c2 + q2)

## 3. Tests

`A` generates odd-length strings whose **centre** is `a`; `B`, centre `b`.

In [ ]:
LA = language(A_, 5)
print("L(A) :", LA)
assert all(len(w) % 2 == 1 and w[len(w)//2] == 'a' for w in LA)
LB = language(B_, 5)
assert all(len(w) % 2 == 1 and w[len(w)//2] == 'b' for w in LB)
print("every member of L(A) has odd length with 'a' dead centre")

The rearrangement really does produce two such groups.

In [ ]:
for s in ['ab', 'aabb', 'abba', 'aabab b'.replace(' ', '')]:
    r = rearrange(s)
    if r is None: continue
    g1, g2 = r
    print("%-8r -> (%r, %r)   lengths %d and %d, total %d"
          % (s, g1, g2, len(g1), len(g2), len(g1) + len(g2)))
    assert len(g1) + len(g2) == len(s)
    assert len(g1) % 2 == 1 and len(g2) % 2 == 1
    assert g1[len(g1)//2] != g2[len(g2)//2]
print("\nboth groups odd-length, with DIFFERENT centre symbols -- so one is an")
print("A and the other is a B, which is why S -> AB | BA.")

Why the regrouping is legal: only the **lengths** were constrained.

In [ ]:
print("original grouping : (p c q)(p' c' q')   with |p|=|p'| and |q|=|q'|")
print("regrouped         : (p c p')(q c' q')   same symbols, same order")
print()
print("The symbols are untouched -- only the bracketing changed.  And the new")
print("bracketing makes each constraint LOCAL: |p|=|p'| is now 'this group is")
print("an odd-length onion around c'.")

So the grammar is correct, and this is how it was found.

In [ ]:
from itertools import product
def in_ww(s): return len(s) % 2 == 0 and s[:len(s)//2] == s[len(s)//2:]
strs = [''.join(p) for k in range(8) for p in product('ab', repeat=k)]
L = set(language(NotWW, 7))
assert L == {s for s in strs if not in_ww(s)}
print("L(NotWW) == complement of L_ww, verified up to length 7")

The idiom, stated for reuse.

In [ ]:
print("When a constraint links distant positions:")
print("  1. write down what is ACTUALLY constrained (usually lengths, not symbols);")
print("  2. look for a regrouping that makes each constraint local;")
print("  3. generate each local group with an onion or a lasso;")
print("  4. join the groups.")
print()
print("That is the whole of 'growing inside-out by rearrangement'.")

## 4. Exercises


1. Do the rearrangement by hand for `aabbab`. Which group is the A?
2. Why must the two centres differ? What would `AA` generate?
3. Apply the same idiom to $\{a^ib^jc^kd^l : i\ne k \text{ or } j\ne l\}$.

In [ ]:
# Your work for the exercises above.